

# Geospatial Indexing in MongoDB
## Advanced Databases - INFO9016: Milestone 3 (tutorial)
---
**Alexandre Dutailly**
Email: alexandre.dutailly@student.uliege.be  
ID: s195855

**Thibaut Lefèvre**
Email: thibaut.lefevre@student.uliege.be  
ID: s164094

---


# 1 Introduction

Geospatial data management and analysis play an essential role in many modern applications, including urban planning, logistics, mobility, interactive mapping and location-based services. Given the increasing volumes of data and the diversity of use cases, it has become crucial to have tools capable of efficiently storing geometric objects (points, lines, polygons, etc.) and quickly executing spatial queries (proximity, inclusion, intersection, etc.).


[MongoDB](https://www.mongodb.com/docs/manual/crud/) offers full native support for the GeoJSON format and 2dsphere indexes, making it easy to implement geospatial processing at scale. Its document-oriented design allows complex entities to be modelled directly, without a rigid schema, and queried using dedicated operators (`$geoWithin`, `$geoIntersects`, `$near`, etc.).


This tutorial describes :

1. **The fundamentals** of the geospatial model in MongoDB: storage formats, indexing and query operators.  
2. **Setting up** a 2dsphere index and importing data from OpenStreetMap (benches, cycle paths, parks).  
3. **The illustration** of queries of increasing complexity: from simple proximity searches to the identification of spatial intersections.  
4. **A critical analysis** of the strengths and limitations of the MongoDB approach for geospatial processing.  


Each step is based on an example and reproducible, so as to provide both a step-by-step guide and an understanding of indexing geospatial data in MongoDB. 

## 2.1 Introduction to MongoDB


MongoDB is a document-oriented NoSQL database designed to store and query semi-structured data in the form of JSON (or, more precisely, BSON) documents. Unlike relational systems, MongoDB does not require a fixed schema: each document can contain a different set of fields, which facilitates the evolution of the data model and rapid iteration during the development phase.


Its distributed architecture is based on :


- **Sharding** to automatically partition collections across multiple nodes, guaranteeing horizontal scalability and scalability.  

- **Replication** via replica sets, ensuring high availability and fault tolerance.  


The richness of its query language (CRUD, aggregation, pipelines) and its multiple integrations (drivers for almost all languages, Kubernetes support, atomic update operators, etc.) make MongoDB a solution that is particularly well suited to web applications, Big Data analyses and, more generally, any use case requiring flexibility, scalability and high availability.




#### Reference for the introduction 2.1 and 1 : 
- [MongoDB and Oracle: Differences and Capabilities – Oracle](https://www.oracle.com/sg/database/mongodb/#:~:text=,the%20distribution%20of%20that%20data)  
  *Presentation of the main differences and capabilities of MongoDB compared with traditional relational databases.*
- [MongoDB Documentation - introduction](https://www.mongodb.com/docs/manual/introduction/)  
  *Introduction of MongoDB with some basic explanations *

## 2.2 Modelling geospatial data

MongoDB uses the [GeoJSON](https://www.mongodb.com/docs/manual/reference/geojson/)  format to represent geometric objects in documents. The supported types are :

- **Point**  
  A single point, defined by a pair of `[longitude, latitude]` coordinates:

```json
{ "type": "Point", "coordinates": [4.397, 50.835] }

- **LineString** \
A line, made up of a list of points.
  ```json
  {
    "type": "LineString",
    "coordinates": [
      [4.392, 50.835],
      [4.395, 50.836],
      [4.400, 50.838]
    ]
  }

- **Polygon**\
A simple polygon, defined by an outer ring and any inner rings (islands). Each ring is a closed list of points (the first and last points are identical, it therefore needs at least 4 pairs of coordinates).
  ```json
  {
    "type": "Polygon",
    "coordinates": [
      [
        [4.390, 50.830],
        [4.405, 50.830],
        [4.405, 50.845],
        [4.390, 50.845],
        [4.390, 50.830]
      ]
    ]
  }


- **Multi** (MultiPoint, MultiLineString, MultiPolygon)\
Groups of several geometries of the same type, to represent several entities in a single field.

#### Reference for 2.2 section.
- [Exploring the Geospatial Capabilities of MongoDB (Medium article)](https://medium.com/@amr.m.m.hassanein/exploring-the-geospatial-capabilities-of-mongodb-sample-code-94a122f572e1#:~:text=1,types)  
  *Article exploring MongoDB's geospatial features.*

## 2.3 Geospatial indexing


MongoDB offers two main types of geospatial index to speed up operations on GeoJSON data:


- **2d** (legacy)  

  Designed for data in plane coordinates (array `[x, y]`), this index only supports simple bounding box queries (`$geoWithin` with `$box`) and the search for neighbouring points (`$near`). It is still useful for datasets projected onto a local plane, but does not handle complex polygons or calculations on a sphere.


- **2dsphere**  

  Built for GeoJSON geometries (Point, LineString, Polygon, Multi*), this index models the Earth as a sphere. It natively optimises :

  - proximity searches (`$near`, `$nearSphere`)  

  - spatial inclusion queries (`$geoWithin` with GeoJSON polygons)  

  - intersections (`$geoIntersects` between lines, polygons and points)  

  - the `$geoNear` stage of the aggregation pipeline


The use of a 2dsphere index is a prerequisite for all applications requiring distance, geofencing or overlay calculations (interactive mapping, fleet tracking, route analysis, location-based services, etc.). Briefly mentioning these indexes informs the reader about the geospatial queries that will really be optimised, as well as the types of applications (localised e-commerce, smart cities, IoT, GIS) that MongoDB can effectively support.




#### Reference for 2.3 section.
- [MongoDB Documentation – Geospatial Index Types](https://www.mongodb.com/docs/manual/core/indexes/index-types/index-geospatial/)  
  *Explanation of the different types of geospatial index supported by MongoDB.*
  
- [MongoDB Documentation – Geospatial Queries](https://www.mongodb.com/docs/manual/geospatial-queries/#:~:text=In%20MongoDB%2C%20you%20can%20store,as%20legacy%20coordinate%20pairs)  
  *Official guide to geospatial queries in MongoDB.*

## 2.4 Geospatial functionality



MongoDB supports, natively and without a fixed schema, all GeoJSON objects (Point, LineString, Polygon and their Multi* variants). Each document can embed its geometry in a field (e.g. `location`), enabling the spatial form, attributes and nested relationships required for mapping or geolocation applications to be grouped together in a single record.



The geospatial engine uses the 2dsphere index to provide a number of key operators:


- **Proximity searches** (`$near`, `$nearSphere`): return the documents closest to a given point, with the distance calculated.  


- **Spatial inclusion** (`$geoWithin`): filter geometries contained entirely within a polygon or box.  


- **Spatial Intersections** (`$geoIntersects`): identify objects that intersect or touch a specified area.  


- **Aggregation Pipeline** (`$geoNear`): integrates a proximity search stage into a pipeline, combining distance, sorting and other transformations (`$match`, `$group`, `$project`, etc.).



Combined with MongoDB's horizontal scalability (sharding) and high availability (replica sets), these features form a powerful foundation for use cases as varied as geofencing, trajectory analysis, map visualisation and real-time location services.  

#### Reference for 2.4 section.
- [MongoDB Documentation – Geospatial Queries](https://www.mongodb.com/docs/manual/geospatial-queries/#:~:text=In%20MongoDB%2C%20you%20can%20store,as%20legacy%20coordinate%20pairs)  
  *Official guide to geospatial queries in MongoDB.*

# 3 Tutorial

## 3.1 Prerequisites and configuration

Before you start, make sure you have the following:

- [Docker Dsktop](https://docs.docker.com/desktop/) to launch MongoDB in isolation.
- [Python 3.8+](https://www.python.org/downloads/) to run the import and analysis scripts.     

To install MongoDB with Docker, assuming the user already has some basic knowledge of MongoDB:

1. Check that Docker is installed and note its version: 
   ```bash
   docker --version

2. Download the official MongoDB image:
    ```bash
    docker pull mongo
    
3. Start a MongoDB container in the background, using port 27017 and changing YOUR_NAME to your session name:
    ```bash
    docker run --name mongodb -d -p 27017:27017 -v C:/Users/<YOUR_NAME>/mongodb_data:/data/db mongo

 Installation of the Python libraries needed for the tutorial :

In [ ]:
!pip install pymongo requests shapely

## 3.2 Downloading OpenStreetMap data

The Overpass API is used to retrieve geographic objects in a given area in JSON format:

- **Benches**  
  Nœuds (`node`) tagged `amenity=bench` (points indicating location of benches).

- **Cycle ways**  
  Ways (`way`) tagged `highway=cycleway` (Lines describing the layout of the cycle ways).

- **Parks**  
  Leisure (`leisure`) tagged `leisure=park` (Lines or polygons representing green areas).

Each entity returned contains its geometry (lat/long coordinates for nodes, lists of points for ways/relations), which can then be viewed or analysed.


In [1]:
import requests

overpass_url = "http://overpass-api.de/api/interpreter"
overpass_query = """
[out:json][timeout:25];
(
  node    ["amenity"="bench"](50.82,4.32,50.88,4.42);
  way     ["highway"="cycleway"](50.82,4.32,50.88,4.42);
  way     ["leisure"="park"](50.82,4.32,50.88,4.42);
);
out geom;
"""
response = requests.get(overpass_url, params={'data': overpass_query})
data = response.json()
print(len(data['elements']))

print(data.keys())


4755
dict_keys(['version', 'generator', 'osm3s', 'elements'])


## 3.3 Connecting to MongoDB and integrating data to the database:

In [ ]:
from pymongo import MongoClient
import requests
from shapely.geometry import shape, mapping

client = MongoClient('mongodb://localhost:27017/')
db = client['geospatial_db']
collection = db['geo_objects']

collection.drop()

docs = []
for elem in data['elements']:
    tags = elem.get('tags', {})

    # Bancs (nodes)
    if elem['type'] == 'node' and tags.get('amenity') == 'bench':
        docs.append({
            'osm_id': elem['id'],
            'tags': tags,
            'location': {
                'type': 'Point',
                'coordinates': [elem['lon'], elem['lat']]
            }
        })

    elif elem['type'] in ('way', 'relation'):
        if tags.get('highway') == 'cycleway' and 'geometry' in elem:
            coords = [[pt['lon'], pt['lat']] for pt in elem['geometry']]
            geom = {'type': 'LineString', 'coordinates': coords}
            
        elif tags.get('leisure') == 'park':
            if 'geometry' in elem:
                coords = [[pt['lon'], pt['lat']] for pt in elem['geometry']]
                if coords[0] != coords[-1]:
                    coords.append(coords[0])
                geom = {'type': 'Polygon', 'coordinates': [coords]}

            elif 'center' in elem:
                lon, lat = elem['center']['lon'], elem['center']['lat']
                geom = {'type': 'Point', 'coordinates': [lon, lat]}

            else:
                continue

        else:
            continue

        docs.append({
            'osm_id': elem['id'],
            'tags': tags,
            'location': geom
        })

# Import dans MongoDB
result = collection.insert_many(docs)
print(f"{collection.count_documents({})} documents insérés")
for doc in collection.find().limit(5):
    print(doc)

## 3.4 Creating the geospatial index and manually inserting a point

After importing our GeoJSON objects into MongoDB (Points, LineStrings, Polygons), we create a 2dsphere index on the `location` field:

In [ ]:
collection.create_index([('location', '2dsphere')])

'location_2dsphere'

In [ ]:
pt = {
    'name': 'Point de test',
    'location': {'type':'Point','coordinates':[4.397,50.835]}
}
id_pt = collection.insert_one(pt).inserted_id

aoi = {
    'type':'Polygon',
    'coordinates':[ [
        [4.390,50.830], [4.405,50.830], [4.405,50.845], [4.390,50.845], [4.390,50.830]
    ]]
}
res = list(collection.find({'location': {'$geoWithin': {'$geometry': aoi}}}))
print(f"Points in the area: {len(res)}")

Points in the area: 173


## 3.5 Using the different types of queries on our collection

In [98]:
# Taking a bench, a park and a cycleway at random, these will be used later in our examples

random_bench = collection.find_one({'tags.amenity':'bench'})
random_park = collection.find_one({'tags.leisure':'park'})
random_cycleway = collection.find_one({'tags.highway': 'cycleway'})

### 3.5.1 Proximity Queries
For this example, we will use two different queries: `$geoNear` and `$nearSphere`.  
- `$nearSphere` returns the elements, sorted from closest to farthest to a given point, under specific conditions (e.g. a maximal distance). It requires geospatial indexes (either 2d or 2dSphere, depending on the type of object). It works similarly to `$near` but is more adapted on 2dSphere indexes (although it can handle 2d indexes).
    - in our example, it is used to find the five closest benches that are located less than 200 meters away from a specific point (here, a random bench).
- `$geoNear`is a bit different, and is an operator that takes part in aggregate pipelines and is therefore not intended to work alone. It returns the distance of an object to a point in ascending order (while checking the different requirements such as the min and max distances)

It has to be noted that these queries only work to compute distances from a point, you can't use it to compute the distance of objects to a polygon. To do this you can for example use the center of the polygon. Another method with polygons will be showed in 3.5.2 inclusion queries.

In [ ]:
# Finding five closest benches that are located less than 200 meters away from a given point, here a random bench we initialised earlier

near_benches = list(collection.find({
    'tags.amenity':'bench',
    # 'tags.highway':'cycleway', # Looking for parks or cycleways both work
    # 'tags.leisure':'park',
    # 'location': {'$nearSphere': {'$geometry': random_park['location'], '$maxDistance': 200}} # Using this would not work as we are not computing distance from a point
    'location': {'$nearSphere': {'$geometry': random_bench['location'], '$maxDistance': 200}}
}).limit(5))
print("5 closest benches located less than 200 meters away from the point:")
for b in near_benches:
    print(b)

For this query, we wanted to get the 5 benches closest to a given point (which also happens to be a bench), but it would also work if we wanted to get the cycleways or the parks closest to this point. On the other hand, we cannot retrieve the 5 closest benches to a cycleway or a park as they are not of type point.

In [ ]:
# Finding the 5 closest elements to a given point and returning their distance to the point

pipeline = [
    {'$geoNear': {
        'near': random_bench['location'],
        'distanceField': 'dist.calculated',
        'spherical': True,

    }},
    {'$limit': 5}
]
res_geo = collection.aggregate(pipeline)
for doc in res_geo:
    print(doc)

Although its usage is usually part of a longer pipeline, we see here that it returns the elements closest to the point but also the distance to the point, which is not the case of `$nearSphere`

### 3.5.2 Inclusion queries  

`$geoWithin` selects objects (of all types) that completely lies into another geospatial object (either polygon or multi polygons). It does not require geospatial indexes   
For this query, we will use two different examples:
- The first one is a rather basic usage of the query and will return the number of benches in each park
- The second one is a bit more specific. It returns the id of benches located within the surrondings of a cycle way, and the id of the cycle way. It looks like a proximity query but remember that those can only be applied to points, and whereas we could reduce a park to its center, it is muche less representative to reduce a cycle way to its center. We therefore use this "trick" to find elements in its surroundings. Notice that it will need the usage of the library `shapely`.

In [ ]:
# Finding the number of bench in each park

# Creating dictionnary with the park names and their number of benches
benches_per_park = {}

for park in collection.find({'tags.leisure': 'park'}):
# for park in collection.find({'tags.amenity': 'bench'}): # Will not work as benches are of type point and not polygon
    
    # counting the number of benches located in the park
    nb_benches = collection.count_documents({
        'tags.amenity': 'bench',
        # 'tags.highway': 'cycleway', # In case we want to retrieve cycleways entirely within parks
        'location': {'$geoWithin': {'$geometry': park['location']}}
    })
    
    # Adding the informations to the dictionnary we created
    if(park['tags'].get('name') is not None):
        benches_per_park[park['tags']['name']] = nb_benches
    else:
        benches_per_park[park['osm_id']] = nb_benches


benches_per_park

In [ ]:
from shapely.geometry import shape, mapping

for cycleway in collection.find({'tags.highway':'cycleway'}):
    poly = shape(cycleway['location']).buffer(0.00005) # Here we take the cycleways and their surrondings with bugffer !
    buffer_geojson = mapping(poly)

    benches = collection.find({
        'tags.amenity': 'bench',
        'location': {
            '$geoWithin': { '$geometry': buffer_geojson }
        }
    })

    for b in benches:
        print(f"Bench {b['osm_id']} lies in the surroundings of cycle way {cycleway['osm_id']}")

### 3.5.3 Intersection
For the example, we want to get the id of the parks a cycle way crosses.  
We use `$geoIntersects` which selects elements that interesect another. This operator does not require a spatial index but does not work with the 2d index. It can be used with all types of objects (although the intersection of objects with points returns no results).

In [ ]:
# Returning the parks a cycle way crosses

for cycleway in collection.find({'tags.highway': 'cycleway'}):
# for cycleway in collection.find({'tags.amenity': 'bench'}): # Also works even though we are using points
    parks = collection.find({
        'tags.leisure': 'park',
        # 'tags.amenity': 'bench', # Does not return anything
        'location': {
            '$geoIntersects': {
                '$geometry': cycleway['location']
            }
        }
    })
    if parks:
        for park in parks:
            print(f"The track {cycleway['osm_id']} crosses the park {park['osm_id']}")
    else:
        print(f"The track {cycleway['osm_id']} crosses no park")

#### Reference for 3. section.
- [MongoDB Documentation – Geospatial Queries](https://www.mongodb.com/docs/manual/geospatial-queries/#:~:text=In%20MongoDB%2C%20you%20can%20store,as%20legacy%20coordinate%20pairs)  
  *Official guide to geospatial queries in MongoDB.*

- [MongoDB Documentation – Geospatial Index Types](https://www.mongodb.com/docs/manual/core/indexes/index-types/index-geospatial/)  
  *Explanation of the different types of geospatial index supported by MongoDB.*

## 4 Advantages/limitations and use cases for the geospatial approach in MongoDB


### 4.1 Advantages


- **Flexible modelling**  
  Thanks to storage in BSON/JSON documents, there is no fixed schema: each entity can embed its attributes and geometry (GeoJSON) without complex migration.  

- **2dsphere index optimised**  
  Designed for the Earth as a sphere, it drastically speeds up distance (`$near`), inclusion (`$geoWithin`) and intersection (`$geoIntersects`) queries, even on large volumes of data.  

- **Integrated aggregation pipeline**  
  The `$geoNear` stage naturally combines with all the stages of the aggregation framework (sorting, grouping, projection), paving the way for very powerful spatial and statistical calculations in the same flow.  




### 4.2 Limitations
 

- **Accuracy and complex geometry**  
  The spherical model is an approximation; for very precise calculations on a very small scale (centimetres), or advanced geometric operations (buffer, complex overlay), a specialised GIS DBMS (PostGIS, Oracle Spatial) is preferable.  

- **Advanced GIS analytical features**  
  MongoDB does not natively provide all topological operations (polygon simplification, advanced area calculation, optimised routes): you will often need to use external libraries (Shapely, GDAL) or a dedicated GIS tool.



#### Reference for 4. section.
- [MongoDB Documentation – Geospatial Index Restrictions](https://www.mongodb.com/docs/manual/core/indexes/index-types/geospatial/restrictions/#:~:text=Geospatial%20indexes%20cannot%20cover%20a,query)  
  *Technical limitations related to geospatial indexes in MongoDB.*

### 4.3 Use cases and real examples


MongoDB geospatial is used in many sectors to take advantage of its spatial data storage and query capabilities. Here are three concrete examples:


- **[Smart Cities (urban management and public services)](https://www.mongodb.com/solutions/customer-case-studies/city-of-chicago)** 
  The city of Chicago has deployed a centralised platform based on MongoDB to aggregate and analyse a wide range of data in real time (transport flows, environmental sensors, citizen reports). Geospatial indexes allow events to be viewed and queried according to their location, optimising infrastructure maintenance and emergency response management.


- **[Internet of Things (IoT) and fleet tracking](https://www.mongodb.com/solutions/customer-case-studies/share-now)**
  SHARE NOW, the car-sharing service created by the merger of car2go and DriveNow, is using MongoDB Atlas to manage real-time data from more than 11,000 vehicles in 16 European cities. Each vehicle sends information such as its location, engine status and fuel level. The microservices connected to MongoDB store and exploit this data to enable predictive maintenance, real-time monitoring of vehicles and optimisation of operations.

- **[Location-based recommendations services](https://www.mongodb.com/solutions/customer-case-studies/system1-mapquest)**   
  Platforms such as MapQuest have integrated MongoDB Atlas to provide suggestions for points of interest and personalised itineraries. By combining `$geoNear` and aggregation pipelines, they automatically suggest the nearest places (petrol stations, restaurants, hotels) based on the user's position, while filtering according to business criteria (opening hours, rating, availability).



## Conclusion

This tutorial has enabled us to explore in detail the geospatial indexing and query capabilities offered by MongoDB. Using concrete examples and real-life use cases (IoT with SHARE NOW and recommendations with MapQuest), we have highlighted the significant advantages of MongoDB, such as the flexibility of the schema, the speed of execution of spatial queries and its ease of integration. However, we have also highlighted certain technical limitations related to geospatial indexes that need to be taken into account during modelling. Ultimately, MongoDB is a wise choice for modern applications that require advanced geospatial functionality combined with high scalability.

## Project process

This project was conducted collaboratively. All phases of the work, from initial research to final writing, were carried out jointly by both of us, ensuring coherence and a consistent shared understanding of the topic.

## AI use 
For the preparation of this report, we used DeepL Translator for translating certain sections into English. \
Additionally, we utilized ChatGPT's deep search feature to assist in identifying sources and to improve the formulation .

# All References :

- [MongoDB and Oracle: Differences and Capabilities – Oracle](https://www.oracle.com/sg/database/mongodb/#:~:text=,the%20distribution%20of%20that%20data)  
  *Presentation of the main differences and capabilities of MongoDB compared with traditional relational databases.*

- [MongoDB Documentation - introduction](https://www.mongodb.com/docs/manual/introduction/)  
  *Introduction of MongoDB with some basic explanations *

- [MongoDB Documentation – Geospatial Queries](https://www.mongodb.com/docs/manual/geospatial-queries/#:~:text=In%20MongoDB%2C%20you%20can%20store,as%20legacy%20coordinate%20pairs)  
  *Official guide to geospatial queries in MongoDB.*

- [MongoDB Documentation – Geospatial Index Types](https://www.mongodb.com/docs/manual/core/indexes/index-types/index-geospatial/)  
  *Explanation of the different types of geospatial index supported by MongoDB.*

- [MongoDB Documentation – Geospatial Index Restrictions](https://www.mongodb.com/docs/manual/core/indexes/index-types/geospatial/restrictions/#:~:text=Geospatial%20indexes%20cannot%20cover%20a,query)  
  *Technical limitations related to geospatial indexes in MongoDB.*

- [Exploring the Geospatial Capabilities of MongoDB (Medium article)](https://medium.com/@amr.m.m.hassanein/exploring-the-geospatial-capabilities-of-mongodb-sample-code-94a122f572e1#:~:text=1,types)  
  *Article exploring MongoDB's geospatial features.*


## Use cases (in the title of this chapter)

[Smart Cities (urban management and public services)](https://www.mongodb.com/solutions/customer-case-studies/city-of-chicago)

[Internet of Things (IoT) and fleet tracking](https://www.mongodb.com/solutions/customer-case-studies/share-now)

[Location-based recommendations services](https://www.mongodb.com/solutions/customer-case-studies/system1-mapquest)

